## Setup

In [ ]:
#Setup DEAP
import os
from deap import base, creator, tools, gp
from experiments import tune_gp_model
from experiments import model_comparisons
from experiments import blackbox_vs_groundtruth
from experiments import compare_two_approaches

#Create results folder
try:
    os.mkdir("results")
except FileExistsError:
    pass

#Set up creator object
creator.create("FitnessMin", base.Fitness, weights=(-1.0,)) #We want to minimise fitness
creator.create("Individual", gp.PrimitiveTree, fitness=creator.FitnessMin) #Individuals are GP trees (with an associated fitness value)


# Experiments

1. Tune baseline GP (no LLM) - tune_gp_model
2. Tune k (plot k) - tune_gp_model
3. Tune self-adapt + default temp (no plot) - tune_gp_model
4. Compare multiple models - model_comparisons(params, names)
5. Black box vs Ground Truth for optimal setup - blackbox_vs_groundtruth(optimal_parameters, standard_model_params, model_name)
6. Compare optimal LLM-GP against Standard GP - compare_two_approaches(dataset, alg1_name, alg2_name, alg1_params, alg2_params, num_runs=10)
7. Compare optimal LLM-GP against GP with Fixed Operator Design - compare_two_approaches(dataset, alg1_name, alg2_name, alg1_params, alg2_params, num_runs=10)
8. Compare Fixed Operator Design GP against Standard GP - compare_two_approaches(dataset, alg1_name, alg2_name, alg1_params, alg2_params, num_runs=10)



## Hyperparmater Tuning

### Baseline Model

In [ ]:
tuning_ranges = {
    "pop_size": [100, 300, 500], #250
    "cxpb": [0.5, 0.7, 0.9],
    "mutpb": [0.01, 0.1, 0.2],
    "tourn_size": [3, 5, 7],
    "k": [1000],
    "self_adapt_req": [None], #Can be set to None (5 works well)
    "default_temperature": [0.3],
    "temperature_alpha": [0.1],
    "model": [None],
    "reasoning_model": [False]
}

tune_gp_model(tuning_ranges, "results/standard_tuning", plot_param=None)


### Tuning K

In [ ]:
#TODO: UPdayte these values
k_ranges = {
    "pop_size": [250], #250
    "cxpb": [0.8],
    "mutpb": [0.1],
    "tourn_size": [3],
    "k": [2, 3, 4, 5, 6, 7],
    "self_adapt_req": [None], #Can be set to None (5 works well)
    "default_temperature": [0.3],
    "temperature_alpha": [0.1],
    "model": [None],
    "reasoning_model": [False]
}

tune_gp_model(k_ranges, "results/k_tuning", plot_param="k")

### Tuning Self-Adaptation Rate

In [ ]:
self_adaptation_ranges = {
    "pop_size": [250], #250
    "cxpb": [0.8],
    "mutpb": [0.1],
    "tourn_size": [3],
    "k": [3],
    "self_adapt_req": [3, 5, 7], #Can be set to None (5 works well)
    "default_temperature": [0.3, 0.5, 0.7],
    "temperature_alpha": [0.1],
    "model": [None],
    "reasoning_model": [False]
}

tune_gp_model(self_adaptation_ranges, "results/self_adaptation", plot_param=None)

## Model Comparison

In [ ]:
models = ["Qwen/Qwen3-Coder-Next-FP8", "zai-org/GLM-5", "moonshotai/Kimi-K2.5", "openai/gpt-oss-120b", "deepseek-ai/DeepSeek-V3.1", "meta-llama/Llama-3.3-70B-Instruct-Turbo", None]
model_names = ["Qwen3-Coder", "GLM-5", "Kimi K2.5", "GPT OSS 120b", "DeepSeek V3.1", "Llama 3.3 70b", "No LLM"]
reasoning = [False, True, True, True, True, False, False]

optimal_parameters = {
    "pop_size": 10, #250
    "gens": 15,
    "max_time": 8.0 * 60.0 * 60.0,
    "cxpb": 0.8,
    "mutpb": 0.1,
    "k": 3, 
    "functions": ["+", "-", "*", "/", "sqrt", "sin", "cos", "log"],
    "verbose": True,
    "self_adapt_req": None, #Can be set to None (5 works well)
    "default_temperature": 0.3,
    "temperature_alpha": 0.1,
    "maximum_stagnation": 10,
    "model": None,
    "reasoning_model": False
}

#Gets parameters for each model
parameters = []
for i in range(len(models)):
    current_parameters = optimal_parameters.copy()
    current_parameters["model"] = models[i]
    current_parameters["reasoning_model"] = reasoning[i]
    parameters.append(current_parameters)

model_comparisons(parameters, model_names)


## Black-box vs Ground Truth Problems

In [ ]:
final_parameters = {
    "pop_size": 10, #250
    "gens": 15,
    "max_time": 8.0 * 60.0 * 60.0,
    "cxpb": 0.8,
    "mutpb": 0.1,
    "k": 3, 
    "functions": ["+", "-", "*", "/", "sqrt", "sin", "cos", "log"],
    "verbose": True,
    "self_adapt_req": None, #Can be set to None (5 works well)
    "default_temperature": 0.3,
    "temperature_alpha": 0.1,
    "maximum_stagnation": 10,
    "model": None,
    "reasoning_model": False
}
model_name = "FILLNAME"

standard_gp_params = {
    "pop_size": 10, #250
    "gens": 15,
    "max_time": 8.0 * 60.0 * 60.0,
    "cxpb": 0.8,
    "mutpb": 0.1,
    "k": 3, 
    "functions": ["+", "-", "*", "/", "sqrt", "sin", "cos", "log"],
    "verbose": True,
    "self_adapt_req": None, #Can be set to None (5 works well)
    "default_temperature": 0.3,
    "temperature_alpha": 0.1,
    "maximum_stagnation": 10,
    "model": None,
    "reasoning_model": False
}

blackbox_vs_groundtruth(optimal_parameters, standard_gp_params, model_name)

## Statistical Analysis

In [ ]:
ground_truth_problems = ["feynman_I_9_18", "feynman_I_6_2a", "feynman_test_10", "feynman_test_10"]
black_box_problems = ["201_pol", "620_fri_c1_1000_25", "1089_USCrime", "4544_GeographicalOriginalofMusic"]
all_problems = ground_truth_problems + black_box_problems

compare_two_approaches(all_problems, "LLM-Based GP", "Standard GP", final_parameters, standard_gp_params, 10)

#TODO: steps 6 and 7